rumz
# Notebook for using Dynamic Time Warping on HSI datasets
### Jun 25
#### V. 1

This notebook processes hsi spectralSum data with DTW to find correlations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import random
import re

### Users should fill in variables with desired core info
Use HSI_July2024 Spreadsheet for ID and section names

Debug variable prints more information during processing

In [2]:
outputDir = '/data/icecores/Output' #png output path

CoreID = "ALHIC2302"
SectionID = "42"

Debug = False
Save = True
SCRIPT_ID = "DTW-NB"

In [3]:
def extract_metadata(filepath: str) -> dict:
    metadata = {}
    in_metadata_section = False
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip().lower() == "## core metadata":
                in_metadata_section = True
                continue
            if in_metadata_section:
                if line.startswith("##"):
                    break
                if '-' in line:
                    try:
                        key, value = line.strip().lstrip("-").split(":", 1)
                        metadata[key.strip()] = value.strip()
                    except ValueError:
                        continue
    return metadata

# Base directory
data_dir = '/data/icecores/Output/AN/ALHIC/ALHIC2302/'
core_section_dirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir)
                     if os.path.isdir(os.path.join(data_dir, d))]

# Dict of paths to valid CSVs: { core_dir : { section_num : [csv paths] } }
valid_csv_paths = {}

for core_dir in core_section_dirs:
    if os.path.basename(core_dir) == "ALHIC2302-34":
        print(f"[i] Skipping excluded core: {core_dir}")
        continue

    timestamped_dirs = [os.path.join(core_dir, d) for d in os.listdir(core_dir)
                        if os.path.isdir(os.path.join(core_dir, d)) and re.match(r'^HSIUM-\d+-\d{4}-10-21_\d{6}$', d)]
    timestamped_dirs = sorted(timestamped_dirs, reverse=True)

    section_paths = {}
    for ts_dir in timestamped_dirs[:6]:  # Limit to most recent 6
        readme_path = os.path.join(ts_dir, 'README.md')
        if not os.path.exists(readme_path):
            continue

        metadata = extract_metadata(readme_path)
        if metadata.get('topLight') == '100' and float(metadata.get('bottomLight', 999)) <= 10:
            for section_num in range(1, 6):
                pattern = f"HSIUM-*_section{section_num}Sums.csv"
                matched_files = sorted(glob.glob(os.path.join(ts_dir, pattern)))
                if matched_files:
                    section_paths[section_num] = matched_files
                else:
                    print(f"[!] Missing section{section_num} CSV in {ts_dir}")
            break  # Use only the first valid timestamp

    if section_paths:
        valid_csv_paths[core_dir] = section_paths
    else:
        print(f"[!] No valid data found for {core_dir}")

# Print result
for core_dir, sections in valid_csv_paths.items():
    print(f"\nCore: {os.path.basename(core_dir)}")
    for section, paths in sections.items():
        print(f"  Section {section}: {len(paths)} file(s)")
        for p in paths:
            print(f"    - {p}")

[!] Missing section1 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12/HSIUM-156-2025-10-21_150005
[!] Missing section2 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12/HSIUM-156-2025-10-21_150005
[!] Missing section3 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12/HSIUM-156-2025-10-21_150005
[!] Missing section4 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12/HSIUM-156-2025-10-21_150005
[!] Missing section5 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12/HSIUM-156-2025-10-21_150005
[!] No valid data found for /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-12
[!] Missing section1 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-36/HSIUM-81-2025-10-21_150005
[!] Missing section2 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-36/HSIUM-81-2025-10-21_150005
[!] Missing section3 CSV in /data/icecores/Output/AN/ALHIC/ALHIC2302/ALHIC2302-36/HSIUM-81-2025-10-21_150005
[!] Missing section4 CSV in /data/icecore

In [4]:
combined_df = None

for core_path, sections in valid_csv_paths.items():
    if f"{CoreID}-{SectionID}" in os.path.basename(core_path):
        section_dfs = []
        for section_num in range(1, 6):
            paths = sections.get(section_num)
            if not paths:
                print(f"[!] Missing CSV for section {section_num}")
                continue

            csv_path = paths[0]  # Assuming one match per section
            try:
                df = pd.read_csv(csv_path, usecols=[0, 1])
                df.columns = ['Depth_cm', f'Strip{section_num}_Sum']
                section_dfs.append(df)
            except Exception as e:
                print(f"[!] Failed to load {csv_path}: {e}")

        if section_dfs and len(section_dfs) == 5:
            # Merge on Depth_cm
            combined_df = section_dfs[0]
            for df in section_dfs[1:]:
                combined_df = pd.merge(combined_df, df, on='Depth_cm')
        else:
            print(f"[!] Incomplete section data for {section_label}")
        break
else:
    print(f"[!] No core found matching section ID {SectionID}")


[!] No core found matching section ID 42


In [5]:
combined_df

In [6]:
from dtaidistance import dtw, dtw_visualisation as dtwvis
from scipy.interpolate import interp1d
from scipy.stats import pearsonr

s1 = combined_df['Strip1_Sum'].values
s2 = combined_df['Strip2_Sum'].values
s3 = combined_df['Strip3_Sum'].values
s4 = combined_df['Strip4_Sum'].values
s5 = combined_df['Strip5_Sum'].values

# Optional: replace NaNs
s1 = np.nan_to_num(s1)
s2 = np.nan_to_num(s2)
s3 = np.nan_to_num(s3)
s4 = np.nan_to_num(s4)
s5 = np.nan_to_num(s5)

distance, paths = dtw.warping_paths(s1, s2)
best_path = dtw.best_path(paths)
similarity_score = distance / len(best_path)

distance1, paths1 = dtw.warping_paths(s2, s3)
best_path1 = dtw.best_path(paths1)
similarity_score1 = distance1 / len(best_path1)

distance2, paths2 = dtw.warping_paths(s3, s4)
best_path2 = dtw.best_path(paths2)
similarity_score2 = distance2 / len(best_path2)

distance3, paths3 = dtw.warping_paths(s4, s5)
best_path3 = dtw.best_path(paths3)
similarity_score3 = distance3 / len(best_path3)

TypeError: 'NoneType' object is not subscriptable

In [ ]:
plt.figure(figsize=(12, 8))

# Original Time Series Plot
ax1 = plt.subplot2grid((2, 2), (0, 0))
ax1.plot(s1, label='Series 1', color='blue')
ax1.plot(s2, label='Series 2', linestyle='--',color='orange')
ax1.set_title('Original Time Series')
ax1.legend()

# Shortest Path Plot (Cost Matrix with the path)
# In this example, only the path is plotted, not the entire cost matrix.

ax2 = plt.subplot2grid((2, 2), (0, 1))
ax2.plot(np.array(best_path)[:, 0], np.array(best_path)[:, 1], 'green', marker='o', linestyle='-')
ax2.set_title('Shortest Path (Best Path)')
ax2.set_xlabel('Series 1')
ax2.set_ylabel('Series 2')
ax2.grid(True)

# Point-to-Point Comparison Plot
ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)
ax3.plot(s1, label='Series 1', color='blue', marker='o')
ax3.plot(s2, label='Series 2', color='orange', marker='x', linestyle='--')
for a, b in best_path:
    ax3.plot([a, b], [s1[a], s2[b]], color='grey', linestyle='-', linewidth=1, alpha = 0.5)
ax3.set_title('Point-to-Point Comparison After DTW Alignment')
ax3.legend()

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

offset = 12000  # vertical spacing between the series
s2_offset = s2 + offset
s3_offset = s3 + (offset * 2)
s4_offset = s4 + (offset * 3)
s5_offset = s5 + (offset * 4)
depths = combined_df["Depth_cm"].values  # assumes same for both series

# Flip so Depth is y-axis
plt.figure(figsize=(6, 10))

# Draw alignment lines
for a, b in best_path:
    plt.plot([s1[a], s2_offset[b]], [depths[a], depths[b]], color='gray', alpha=0.5, linewidth=0.5)
for a, b in best_path1:
    plt.plot([s2_offset[a], s3_offset[b]], [depths[a], depths[b]], color='gray', alpha=0.5, linewidth=0.5)
for a, b in best_path2:
    plt.plot([s3_offset[a], s4_offset[b]], [depths[a], depths[b]], color='gray', alpha=0.5, linewidth=0.5)
for a, b in best_path3:
    plt.plot([s4_offset[a], s5_offset[b]], [depths[a], depths[b]], color='gray', alpha=0.5, linewidth=0.5)

# Plot Series
plt.plot(s1, depths, label='Strip 1', color='blue', marker='.', markersize=2)
plt.plot(s2_offset, depths, label='Strip 2 + offset', color='orange', marker='x', linestyle='--', markersize=2)
plt.plot(s3_offset, depths, label='Strip 3 + offset', color='green', marker='.', markersize=2)
plt.plot(s4_offset, depths, label='Strip 4 + offset', color='red', marker='x', linestyle='--', markersize=2)
plt.plot(s5_offset, depths, label='Strip 5 + offset', color='purple', marker='.', markersize=2)

plt.gca().invert_yaxis()  # Depth increases downward

plt.ylabel('Depth (cm)')
plt.title('DTW Alignment with Visual Offset (Vertical)')
plt.legend()
plt.tight_layout()
plt.show()